# コイン集めアリーナ — クラス対戦演習

これまでの発展教材（クラス → 簡単なゲーム → バイナリとセキュリティ → ネットワークプログラミング）の 総まとめ です。

教室のみなさんのPython botが、1つの盤面で同時に競います。

## ゲームのルール

- 盤面（24×24マス）に コイン がいくつか出現します
- 各プレイヤーは、上下左右に1マスずつ動けます
- コインの上に乗ると +1点。コインは別の場所に再出現します
- 制限時間で、得点が一番高い人の勝ち

## 2つの役割

- 先生: サーバ（審判＋盤面）を立て、公開URL を配り、盤面をスクリーンに映す
- 生徒: サーバに接続する bot を書く。良い戦略を考えた人が勝つ

使う技術は全部これまでの復習です：HTTPクライアント/サーバ・JSON・bot（ネットワーク回）、最後の発展で ハッシュ／署名（バイナリ回）も出てきます。

---
# 共通: アリーナのコード

次の 2つのセル（盤面の見た目 と サーバ本体）は、先生、および ソロ練習する生徒 が実行します。アリーナを動かすための土台です。

まず下のセルで、盤面の見た目（JavaScript）を用意します。折りたたまれています。中身のJS（盤面の見た目）に興味があれば、タイトルをクリックすると展開できます。

In [ ]:
#@title 盤面の見た目 { display-mode: "form" }
# 盤面のHTML/JSを board.html に書き出す（サーバがこれを配信する）
board_html = r'''
<!doctype html><html><head><meta charset="utf-8"><title>コイン集めアリーナ</title>
<style>
 body{background:#0d1117;color:#e6edf3;font-family:sans-serif;text-align:center;margin:0}
 canvas{background:#161b22;border-radius:8px;margin-top:10px}
 #rank{display:inline-block;text-align:left;margin:8px auto;font-size:15px}
</style></head>
<body>
 <h2>コイン集めアリーナ</h2>
 <canvas id="c" width="480" height="480"></canvas>
 <div id="rank"></div>
 <script>
   const cell = 20;
   const ctx = document.getElementById('c').getContext('2d');
   async function tick() {
     let s;
     try { s = await (await fetch('/state')).json(); } catch (e) { return; }
     ctx.clearRect(0, 0, 480, 480);
     // 格子線
     ctx.strokeStyle = '#2a2f3a';
     ctx.lineWidth = 1;
     for (let i = 0; i <= 24; i++) {
       ctx.beginPath(); ctx.moveTo(i * cell, 0); ctx.lineTo(i * cell, 480); ctx.stroke();
       ctx.beginPath(); ctx.moveTo(0, i * cell); ctx.lineTo(480, i * cell); ctx.stroke();
     }
     // コイン（黄色）
     ctx.fillStyle = '#f1c40f';
     for (const c of s.coins) {
       ctx.beginPath();
       ctx.arc(c[0] * cell + cell / 2, c[1] * cell + cell / 2, 6, 0, 7);
       ctx.fill();
     }
     // プレイヤー（各自の色）
     for (const p of s.players) {
       ctx.fillStyle = p.color;
       ctx.beginPath();
       ctx.arc(p.x * cell + cell / 2, p.y * cell + cell / 2, 8, 0, 7);
       ctx.fill();
       ctx.fillStyle = '#fff';
       ctx.font = '10px sans-serif';
       ctx.fillText(p.name, p.x * cell - 4, p.y * cell - 4);
     }
     // ランキング
     const r = [...s.players].sort((a, b) => b.score - a.score);
     document.getElementById('rank').innerHTML = '<b>ランキング</b><br>' +
       r.map((p, i) => `${i + 1}. <span style="color:${p.color}">■</span> ${p.name}: ${p.score}`).join('<br>');
   }
   setInterval(tick, 150);
 </script>
</body></html>
'''
with open("board.html", "w", encoding="utf-8") as f:
    f.write(board_html)
print("board.html を用意しました。")

次がサーバ本体です。ネットワーク回で学んだ `http.server` の応用なので、興味があれば読んでみてください。

In [ ]:
# === アリーナサーバ（ネットワーク回で学んだ http.server の応用）===
# ※ 先に「board.html を書き出すセル」を実行しておくこと
import json, threading, time, random
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

W, H, NUM_COINS = 24, 24, 6

def _rand_pos(): return [random.randint(0, W-1), random.randint(0, H-1)]

class ArenaServer(ThreadingHTTPServer):
    """世界の状態（プレイヤー・コイン）を、このサーバ自身が持つ。
       別サーバ（本番とソロ練習）は状態を共有しない。"""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.lock = threading.Lock()
        self.players = {}
        self.coins = [_rand_pos() for _ in range(NUM_COINS)]
        self.next_id = 0
    def reset(self):
        with self.lock:
            self.players = {}
            self.coins = [_rand_pos() for _ in range(NUM_COINS)]
            self.next_id = 0

class Arena(BaseHTTPRequestHandler):
    def _send(self, obj, code=200):
        b = json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Access-Control-Allow-Origin", "*")   # どこからでも接続可
        self.end_headers(); self.wfile.write(b)
    def _body(self):
        n = int(self.headers.get("Content-Length", 0))
        return json.loads(self.rfile.read(n) or b"{}")

    def do_GET(self):
        srv = self.server
        if self.path == "/" or self.path.startswith("/?"):
            with open("board.html", encoding="utf-8") as f:
                body = f.read().encode()
            self.send_response(200)
            self.send_header("Content-Type", "text/html; charset=utf-8")
            self.end_headers(); self.wfile.write(body)
        elif self.path == "/state":
            with srv.lock:
                self._send({"w": W, "h": H, "coins": srv.coins,
                    "players": [{"id": i, "name": p["name"], "x": p["x"], "y": p["y"],
                                 "score": p["score"], "color": p["color"]} for i, p in srv.players.items()]})
        else:
            self._send({"error": "not found"}, 404)

    def do_POST(self):
        srv = self.server
        d = self._body()
        if self.path == "/join":
            with srv.lock:
                i = srv.next_id; srv.next_id += 1; pos = _rand_pos()
                srv.players[i] = {"name": str(d.get("name", "noname"))[:12], "x": pos[0], "y": pos[1],
                                  "score": 0, "color": "#%06x" % random.randint(0x333333, 0xffffff)}
                self._send({"id": i, "w": W, "h": H})
        elif self.path == "/move":
            with srv.lock:
                p = srv.players.get(d.get("id"))
                if not p: self._send({"error": "unknown id"}, 400); return
                dx, dy = {"up": (0,-1), "down": (0,1), "left": (-1,0), "right": (1,0)}.get(d.get("dir"), (0,0))
                p["x"] = max(0, min(W-1, p["x"] + dx)); p["y"] = max(0, min(H-1, p["y"] + dy))
                for c in srv.coins:
                    if c[0] == p["x"] and c[1] == p["y"]:
                        p["score"] += 1; c[0], c[1] = _rand_pos()
                self._send({"x": p["x"], "y": p["y"], "score": p["score"]})
        elif self.path == "/reset":
            srv.reset()
            self._send({"ok": True})
        else:
            self._send({"error": "not found"}, 404)

    def log_message(self, *a): pass

def start_server():
    srv = ArenaServer(("0.0.0.0", 0), Arena)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    time.sleep(0.3)
    return srv, srv.server_address[1]

print("アリーナの準備OK。start_server() で起動できます。")

---
# A. 先生セクション

先生だけが実行します。サーバを起動し、公開URLを発行し、盤面をスクリーンに映します。

In [ ]:
# 【先生】サーバを起動する
server, PORT = start_server()
print("サーバ起動 ポート:", PORT)

## 公開URLを発行する

`cloudflared` で、教室の全員が接続できる 公開URL（`https://xxxx.trycloudflare.com`）を発行します。

In [ ]:
#@title 公開URLを発行（cloudflared） { display-mode: "form" }
# ※ ネットワーク環境によっては使えないことがあります。その場合は「ソロ練習」で開発を。
import os, re, subprocess, urllib.request

if not os.path.exists("cloudflared"):
    print("cloudflared をダウンロード中...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared")
    os.chmod("cloudflared", 0o755)

proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:%d" % PORT],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for line in proc.stdout:                      # 出力から公開URLを拾う
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0); break
print("=" * 50)
print("公開URL:", public_url)
print("=" * 50)

## ラウンドをリセットする

対戦を仕切り直すとき（全員の得点・位置を消して新しいコインにする）に実行します。

In [ ]:
# 【先生】ラウンドをリセット
import requests
requests.post("http://127.0.0.1:{}/reset".format(PORT))
print("アリーナをリセットしました（新しいラウンド）")

---
# B. 生徒セクション

各自のColabで、サーバに接続する bot を作ります。

## APIの仕様

| やりたいこと | 呼び出し | 返り値 |
|---|---|---|
| 参加する | `POST /join` `{"name": "名前"}` | `{"id": 自分の番号, "w":24, "h":24}` |
| 今の盤面を見る | `GET /state` | `{"coins": [[x,y]...], "players": [{"id","name","x","y","score","color"}...]}` |
| 動く | `POST /move` `{"id": 自分の番号, "dir": 向き}` | `{"x","y","score"}` |

- 向き `dir` は `"up"` / `"down"` / `"left"` / `"right"`
- 座標は x=右方向、y=下方向（左上が 0,0）

## 接続設定

サーバとのやり取りは、ネットワーク回で習った `requests` で行います。使う API は3つだけです。

- 参加する: `requests.post(SERVER + "/join", json={"name": 名前}).json()`
- 盤面を見る: `requests.get(SERVER + "/state").json()`
- 動く: `requests.post(SERVER + "/move", json={"id": 自分のid, "dir": 向き}).json()`

まず接続先の変数 `SERVER` を用意します。

In [ ]:
import requests

# 接続先。この後の「ローカルで練習」のセルで自動的に設定されます。
SERVER = ""

## まずローカルで練習する

本番（先生の公開URL）につなぐ前に、自分のColabの中にアリーナを立てて練習します。下のセルを実行すると、接続先 `SERVER` が自分のローカルサーバに変わり、ダミーの敵も動き出します。

In [ ]:
#@title ソロ練習の準備（サーバ＋ダミー敵） { display-mode: "form" }
# 先に「board.html」「アリーナサーバ」「接続設定」のセルを実行しておくこと
import threading, time, random, requests

srv_local, solo_port = start_server()
SERVER = "http://127.0.0.1:%d" % solo_port   # 接続先を自分のローカルサーバに切り替え
print("ローカルのアリーナ:", SERVER)

def dummy(name):                         # ランダムに動くだけのダミー敵
    m = requests.post(SERVER + "/join", json={"name": name}).json(); mid = m["id"]
    while True:
        requests.post(SERVER + "/move", json={"id": mid, "dir": random.choice(["up", "down", "left", "right"])})
        time.sleep(0.15)
for nm in ["敵A", "敵B", "敵C"]:
    threading.Thread(target=dummy, args=(nm,), daemon=True).start()
print("ダミーの敵を3体 動かしました。")

# 盤面をノート内に表示（Colab専用）
try:
    from google.colab.output import serve_kernel_port_as_iframe
    serve_kernel_port_as_iframe(solo_port, path="/")
except Exception:
    pass

print("この状態で、下の『botを作る』の例題セルを実行すると対戦できます。")

## join：ゲームに参加する

サーバに参加すると、自分の `id` がもらえます。これから、この id で「自分」を指定します。まず例題を実行し、次の演習を自分で書きましょう。

In [ ]:
# 例題：参加して、自分の id をもらう
me = requests.post(SERVER + "/join", json={"name": "テスト"}).json()
print(me)                 # 例: {'id': 0, 'w': 24, 'h': 24}
my_id = me["id"]
print("あなたの id:", my_id)

In [ ]:
# 演習：好きな名前で参加して、自分の id を表示しよう



## state：今の盤面を見る

`GET /state` で、コインの位置と全プレイヤーの状態が返ります。

- `state["coins"]` … `[[x, y], ...]`
- `state["players"]` … `[{"id","name","x","y","score","color"}, ...]`

In [ ]:
# 例題：今の盤面を取得して中身を見る
state = requests.get(SERVER + "/state").json()
print("コインの数:", len(state["coins"]))
print("プレイヤー:", state["players"])

In [ ]:
# 演習：state を取得して、今この盤面に何人プレイヤーがいるか表示しよう



## move：動く

自分の `id` と向き（`"up"` / `"down"` / `"left"` / `"right"`）を送ると、その方向へ1歩動きます。コインに乗ると得点です。

In [ ]:
# 例題：右に1歩動く
result = requests.post(SERVER + "/move", json={"id": my_id, "dir": "right"}).json()
print(result)             # 例: {'x': 11, 'y': 5, 'score': 0}

In [ ]:
# 演習：for ループで、好きな向きに5歩動かしてみよう



## botを作る

join / state / move が分かれば、あとは「参加 → くり返し（盤面を見て、動く）」でbotになります。下の例題は、いつも一番近いコインへ向かうbotです（15秒間プレイ）。まず動かしてみましょう。

In [ ]:
# 例題：一番近いコインへ向かうbot（15秒プレイ）
import time

me = requests.post(SERVER + "/join", json={"name": "あなた"}).json()
my_id = me["id"]

end = time.time() + 15
while time.time() < end:
    state = requests.get(SERVER + "/state").json()

    # 自分を探す
    me_now = None
    for p in state["players"]:
        if p["id"] == my_id:
            me_now = p

    # 一番近いコインを探す
    best = None
    best_dist = 9999
    for c in state["coins"]:
        dist = abs(c[0] - me_now["x"]) + abs(c[1] - me_now["y"])
        if dist < best_dist:
            best_dist = dist
            best = c

    # そのコインの方へ動く向きを決める
    if best[0] > me_now["x"]:
        direction = "right"
    elif best[0] < me_now["x"]:
        direction = "left"
    elif best[1] > me_now["y"]:
        direction = "down"
    else:
        direction = "up"

    requests.post(SERVER + "/move", json={"id": my_id, "dir": direction})
    time.sleep(0.1)

# 結果を表示
score = 0
for p in requests.get(SERVER + "/state").json()["players"]:
    if p["id"] == my_id:
        score = p["score"]
print("15秒プレイ終了。得点:", score)

## botを強くする（演習）

上の例題をコピーして、自分なりに強くしてみましょう。たとえば:

- 他プレイヤーがすぐ近くにいるコインは避ける（`state["players"]` を見る）
- コインが密集している方を狙う

くり返しの中の「向きを決める部分」を工夫します。

In [ ]:
# 上の例題をコピーして、向きの決め方を工夫しよう



---
## 本番：クラス対戦

完成した自分のbotで、先生の公開URLにつないで全員と競います。下のセルで接続先を本番に変えてから、自分のbotのセルをもう一度実行しましょう。

In [ ]:
# 本番：先生の公開URLに変えて、自分のbotで参加する
SERVER = "https://ここに先生のURL.trycloudflare.com"

# SERVER を変えたら、上で作った自分のbotのセルをもう一度実行すると本番に参加できます。
# 対戦時間に合わせて、bot の中の end = time.time() + 15 の 15 を長くしてもOKです。
print("接続先を本番に変えました。自分のbotのセルを実行しましょう。")

---
# 不正（チート）とその対策

このアリーナは、実は id さえ知っていれば他人のプレイヤーも動かせて しまいます。通信は前回学んだとおり HTTPで丸見え なので、覗いて真似すれば「なりすまし移動」ができてしまうのです。

「不正ができてしまう」と分かるのは、仕組みを理解できた証拠。では、どう防ぐか——前回の ハッシュ／デジタル署名 の出番です。

---
# 発展: 人間プレイヤーとして参加する（Gradio操作パネル）

これまでプレイヤーは bot（プログラム） でした。前回学んだ Gradio で 操作パネル を作れば、人間がボタンで参加して、みんなのbotに混じって遊べます。

仕組みは数当ての公開と同じ「Gradio ← `requests` → アリーナサーバ」。ボタンを押すと `/move` をサーバへ送り、`/state` を取り直して盤面を絵にして表示します。

- 盤面を絵に描く `render()` は用意してあります（`PIL`）
- あなたが書くのは、ボタンが押されたときに サーバへ動きを送る 1行です

先に「接続設定」（`SERVER` と `api`）と、対戦相手（先生のサーバ / ソロ練習）を用意しておいてください。

In [ ]:
!pip install --quiet gradio

In [ ]:
import gradio as gr
import requests
from PIL import Image, ImageDraw

CELL = 16   # 1マスの大きさ（ピクセル）

def render(state):
    """盤面の状態を絵にする"""
    W, H = state["w"], state["h"]
    img = Image.new("RGB", (W * CELL, H * CELL), "#161b22")
    d = ImageDraw.Draw(img)
    for i in range(W + 1):                             # 格子線
        d.line([(i*CELL, 0), (i*CELL, H*CELL)], fill="#2a2f3a")
    for j in range(H + 1):
        d.line([(0, j*CELL), (W*CELL, j*CELL)], fill="#2a2f3a")
    for x, y in state["coins"]:                        # コイン（黄）
        d.ellipse([x*CELL+4, y*CELL+4, x*CELL+CELL-4, y*CELL+CELL-4], fill="#f1c40f")
    for p in state["players"]:                         # プレイヤー（各色）
        x, y = p["x"], p["y"]
        d.ellipse([x*CELL+2, y*CELL+2, x*CELL+CELL-2, y*CELL+CELL-2], fill=p["color"])
    return img

my_id = None

def join_game(name):
    global my_id
    my_id = requests.post(SERVER + "/join", json={"name": name}).json()["id"]   # 参加して自分のidをもらう
    return render(requests.get(SERVER + "/state").json()), "参加しました（id={}）".format(my_id)

def move(direction):
    if my_id is None:
        return None, "先に「参加する」を押してください"

    # ここで、direction 方向へ /move する（requests で SERVER に送る）


    state = requests.get(SERVER + "/state").json()     # 最新の盤面を取り直す
    me = next((p for p in state["players"] if p["id"] == my_id), None)
    return render(state), "あなたのスコア: {}".format(me["score"] if me else 0)

# --- 操作パネルの見た目 ---
with gr.Blocks() as demo:
    gr.Markdown("## アリーナ操作パネル（人間プレイヤー）")
    name_in = gr.Textbox(label="名前", value="人間")
    join_btn = gr.Button("参加する", variant="primary")
    board = gr.Image(label="盤面")
    info = gr.Textbox(label="状態")
    up = gr.Button("↑ 上")
    with gr.Row():
        left = gr.Button("← 左"); down = gr.Button("↓ 下"); right = gr.Button("→ 右")

    join_btn.click(join_game, inputs=name_in, outputs=[board, info])
    up.click(lambda: move("up"),       outputs=[board, info])
    down.click(lambda: move("down"),   outputs=[board, info])
    left.click(lambda: move("left"),   outputs=[board, info])
    right.click(lambda: move("right"), outputs=[board, info])

demo.launch(share=True)

In [ ]:
# 実験：このサーバは id さえ知っていれば「他人のプレイヤーも動かせて」しまう（認証が無いため）
import requests
try:
    print(requests.post(SERVER + "/move", json={"id": 0, "dir": "up"}).json())   # 他人(id=0)でも動いてしまうかも
except Exception as e:
    print("エラー:", e)

# 対策の考え方（前回の「ハッシュ／署名」の応用）:
#  1) join のときサーバが各プレイヤーに「秘密トークン」を配る
#  2) move には (id, dir, トークンから作った署名) を付けて送る
#  3) サーバは署名を検証し、正しい持ち主だけ動かす
# → 通信が丸見えでも、署名は本人しか作れないので「なりすまし移動」を防げる。

---
# おわりに

ここまでで、あなたは自分の手で サーバを立て、通信し、botを動かし、全員で競い、そして不正と対策まで 体験しました。

- クラスでデータの設計図を作り
- バイナリとセキュリティでデータの正体と守り方を知り
- ネットワークで通信の仕組みを作り
- そして今日、それらを 全部つないで 動くものを作りました

ここからは、ルールを変える・戦略を磨く・不正対策を実装する——好きに拡張してみてください。おつかれさまでした。